本 notebook 是 [M04.1 统计学习基础](../notes/01-statistical-learning.qmd) 的实例讲解。

1. 1. 经验风险与候选函数选择
2. 2. 多训练集的偏差–方差分解
3. 3. 正则化与样本量的共同作用
4. 4. 有限假设类的统一界

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

In [1]:
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from numpy.polynomial import Polynomial
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. 经验风险与候选函数选择

用固定有限候选集近似 ERM，同时与最小二乘库解对照，分清候选类限制与优化误差。

In [2]:
x = rng.uniform(-1, 1, 80)
y = 1.7 * x + 0.4 + rng.normal(0, 0.2, len(x))
slopes = np.linspace(-1, 3, 81)
intercepts = np.linspace(-1, 1, 81)
loss = np.mean(
    (y[None, None, :] - slopes[:, None, None] * x - intercepts[None, :, None]) ** 2, axis=2
)
i, j = np.unravel_index(np.argmin(loss), loss.shape)
model = LinearRegression().fit(x[:, None], y)
print("网格 ERM:", slopes[i], intercepts[j], loss[i, j])
print("连续最小二乘:", model.coef_[0], model.intercept_)
print("已知生成系数:", 1.7, 0.4)
assert abs(slopes[i] - model.coef_[0]) <= 0.03

网格 ERM: 1.75 0.375 0.034169081724205755
连续最小二乘: 1.7465440657396647 0.3747454817349459
已知生成系数: 1.7 0.4


读法：网格最优只在给定离散类中最优；扩大搜索精度只减少优化近似，不会消除抽样噪声或错误模型类带来的偏差。

## 2. 多训练集的偏差–方差分解

固定测试网格，重新抽取训练集和噪声，对同一学习算法重复拟合，数值核验平方误差分解。

In [3]:
grid = np.linspace(-0.9, 0.9, 120)
truth = np.sin(np.pi * grid)
sigma = 0.2
rows = []
for degree in [1, 3, 12]:
    predictions = []
    for _ in range(120):
        x = rng.uniform(-1, 1, 35)
        y = np.sin(np.pi * x) + rng.normal(0, sigma, len(x))
        predictions.append(Polynomial.fit(x, y, degree, domain=[-1, 1])(grid))
    predictions = np.array(predictions)
    mean = predictions.mean(axis=0)
    bias2 = np.mean((mean - truth) ** 2)
    variance = np.mean(predictions.var(axis=0))
    mse_to_truth = np.mean((predictions - truth) ** 2)
    print("分解残差:", degree, mse_to_truth - bias2 - variance)
    rows.append([degree, bias2, variance, sigma * sigma, mse_to_truth + sigma * sigma])
    assert np.isclose(mse_to_truth, bias2 + variance, atol=1e-10)
print(
    pd.DataFrame(
        rows, columns=["degree", "bias squared", "variance", "noise", "expected noisy risk"]
    )
)

分解残差: 1 1.734723475976807e-18
分解残差: 3 -1.734723475976807e-18
分解残差: 12 0.0
   degree  bias squared   variance  noise  expected noisy risk
0       1      0.151520   0.015484   0.04             0.207004
1       3      0.003800   0.004830   0.04             0.048630
2      12      0.126338  19.187062   0.04            19.353400


读法：方差来自重新抽取训练集，不是同一次拟合的残差方差；这里报告有限重复的分解，并加上已知独立标签噪声的解析期望。

## 3. 正则化与样本量的共同作用

在固定高阶表示下用 Ridge 收缩系数，比较训练误差与独立网格的无噪声误差；不在该网格上选超参数。

In [4]:
rows = []
test_x = np.linspace(-1, 1, 500)[:, None]
test_y = np.sin(np.pi * test_x[:, 0])
for n in [20, 60, 180]:
    x = rng.uniform(-1, 1, (n, 1))
    y = np.sin(np.pi * x[:, 0]) + rng.normal(0, 0.2, n)
    for alpha in [0.0001, 0.1, 10.0]:
        model = make_pipeline(
            PolynomialFeatures(12, include_bias=False), StandardScaler(), Ridge(alpha=alpha)
        )
        model.fit(x, y)
        rows.append(
            [
                n,
                alpha,
                np.mean((y - model.predict(x)) ** 2),
                np.mean((test_y - model.predict(test_x)) ** 2),
            ]
        )
print(pd.DataFrame(rows, columns=["n", "alpha", "training MSE", "truth-grid MSE"]))

     n    alpha  training MSE  truth-grid MSE
0   20   0.0001      0.028640        0.689748
1   20   0.1000      0.043872        0.032094
2   20  10.0000      0.197565        0.288096
3   60   0.0001      0.035666        0.004408
4   60   0.1000      0.039274        0.005137
5   60  10.0000      0.133716        0.101023
6  180   0.0001      0.040352        0.004032
7  180   0.1000      0.041708        0.003462
8  180  10.0000      0.085165        0.037952


读法：正则化改变学习算法及其偏差方差，样本量改变估计波动。单个种子的一张表不能证明固定 alpha 总是最好，正式选择需要训练侧验证。

## 4. 有限假设类的统一界

对固定的有限预测函数同时估计有界分类风险，将经验最大偏差与 Hoeffding 加并合界得到的上界比较。

In [5]:
n = 600
thresholds = np.linspace(0, 1, 31)
delta = 0.05
x = rng.uniform(size=n)
y = x > 0.43
empirical = np.mean((x[None, :] > thresholds[:, None]) != y, axis=1)
exact = np.abs(thresholds - 0.43)
epsilon = np.sqrt(np.log(2 * len(thresholds) / delta) / (2 * n))
print("最大经验偏差:", np.max(np.abs(empirical - exact)), "统一界 epsilon:", epsilon)
print("候选函数数量:", len(thresholds), "样本量:", n)
print("观测到是否落在界内:", bool(np.max(np.abs(empirical - exact)) <= epsilon))

最大经验偏差: 0.031666666666666676 统一界 epsilon: 0.07704363838435485
候选函数数量: 31 样本量: 600
观测到是否落在界内: True


读法：界依赖独立同分布样本、有界损失和预先固定的有限函数类；一次落在界内不是对高概率定理的实验证明。无限神经网络类不能直接代入参数个数替代假设数。

## 小结

统计学习把函数偏好、有限数据和未来风险连接起来。训练目标可精确计算，泛化风险需要抽样与评估假设；偏差方差解释重复拟合的波动，正则化表达有方向的偏好。下一步是在具体模型中检验这些取舍，而不是只追求更低的训练损失。